# 3 — Análisis por Familia de Features

Las features se agrupan en familias con significado físico distinto:

| Familia | Prefijo | Descripción física |
|---------|---------|--------------------|
| **Magnitud** | `mag_*` | Estadísticos de la envolvente de la señal |
| **Fase** | `phase_*` | Estadísticos de la fase (circular, unwrap, saltos) |
| **PDP** | `pdp_*` | Power Delay Profile — dispersión temporal del canal |
| **SVD** | `svd_*` | Descomposición en valores singulares de la matriz de canal |
| **Doppler** | `doppler_*` | Spread Doppler — dispersión en frecuencia |
| **dH/dt** | `dH_dt_*` | Variación temporal del canal |

**Objetivo:** entrenar un RF con cada familia por separado y comparar MAE y R² para determinar qué fenómeno físico porta más información sobre la temperatura.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score

CSV_PATH  = Path("dataset_features_temperatura.csv")
FEAT_JSON = Path("features_seleccionadas.json")

In [ ]:
df = pd.read_csv(CSV_PATH)

META_COLS = {
    "sample_uid","sample_id","tx_path","rx_path",
    "tx_path_rel_to_bbdd_parent","rx_path_rel_to_bbdd_parent",
    "split","temperature_folder","N_subcarriers","M_symbols","num_valid","valid_ratio",
}
CONSTANT_COLS = {"phase_jump_m_count","phase_jump_m_ratio","phase_jump_k_count","phase_jump_k_ratio"}

ALL_FEAT = [c for c in df.columns if c not in META_COLS and c != "temperature" and c not in CONSTANT_COLS]
print(f"Total features: {len(ALL_FEAT)}")
print(ALL_FEAT)

In [ ]:
# ── Definición de familias ────────────────────────────────────────────────────
FAMILIES = {
    "Magnitud"  : [f for f in ALL_FEAT if f.startswith("mag_")],
    "Fase"      : [f for f in ALL_FEAT if f.startswith("phase_")],
    "PDP"       : [f for f in ALL_FEAT if f.startswith("pdp_")],
    "SVD"       : [f for f in ALL_FEAT if f.startswith("svd_")],
    "Doppler"   : [f for f in ALL_FEAT if f.startswith("doppler_")],
    "dH/dt"     : [f for f in ALL_FEAT if f.startswith("dH_dt_")],
    "Todas"     : ALL_FEAT,  # referencia completa
}
for name, feats in FAMILIES.items():
    print(f"{name:12s}: {len(feats):3d} features  → {feats}")

In [ ]:
# Splits
df_tr = df[df["split"] == "train"]
df_te = df[df["split"] == "test"]

def fit_eval(feats):
    if not feats:
        return None
    X_tr = df_tr[feats].values; y_tr = df_tr["temperature"].values
    X_te = df_te[feats].values; y_te = df_te["temperature"].values
    pipe = Pipeline([("imp", SimpleImputer(strategy="mean")),
                     ("m",   RandomForestRegressor(n_estimators=100, min_samples_leaf=2, n_jobs=-1, random_state=42))])
    pipe.fit(X_tr, y_tr)
    preds = pipe.predict(X_te)
    return {
        "mae":   mean_absolute_error(y_te, preds),
        "r2":    r2_score(y_te, preds),
        "preds": preds,
        "y_te":  y_te,
    }

results = {}
for name, feats in FAMILIES.items():
    print(f"Entrenando {name:12s} ({len(feats)} feats)...", end=" ", flush=True)
    res = fit_eval(feats)
    if res:
        results[name] = res
        print(f"MAE={res['mae']:.3f}°C  R2={res['r2']:.4f}")
    else:
        print("sin features")

In [ ]:
# ── Tabla resumen ──────────────────────────────────────────────────────────────
rows = [{"Familia": n, "Nº features": len(FAMILIES[n]),
         "MAE [°C]": round(r["mae"],3), "R²": round(r["r2"],4)}
        for n, r in results.items()]
df_res = pd.DataFrame(rows).sort_values("MAE [°C]")
print(df_res.to_string(index=False))

In [ ]:
# ── Figura: MAE y R² por familia ──────────────────────────────────────────────
names = [r["Familia"] for _, r in df_res.iterrows()]
maes  = [r["MAE [°C]"] for _, r in df_res.iterrows()]
r2s   = [r["R²"]       for _, r in df_res.iterrows()]
colors = ["#C00000" if n == "Todas" else "#2E75B6" for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.barh(names, maes, color=colors, alpha=0.85)
for b, v in zip(bars, maes):
    ax.text(v + 0.02, b.get_y()+b.get_height()/2, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("MAE [°C]", fontsize=10)
ax.set_title("MAE por familia (menor = mejor)", fontsize=11, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
import matplotlib.patches as mpatches
ax.legend(handles=[mpatches.Patch(color="#C00000", label="Todas las features"),
                   mpatches.Patch(color="#2E75B6", label="Una familia")], fontsize=9)

ax = axes[1]
bars = ax.barh(names, r2s, color=colors, alpha=0.85)
for b, v in zip(bars, r2s):
    ax.text(v + 0.002, b.get_y()+b.get_height()/2, f"{v:.4f}", va="center", fontsize=9)
ax.set_xlabel("R²", fontsize=10)
ax.set_title("R² por familia (mayor = mejor)", fontsize=11, fontweight="bold")
ax.grid(axis="x", alpha=0.3)

plt.suptitle("Capacidad predictiva de cada familia de features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("familias_mae_r2.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Scatter predicho vs real por familia ─────────────────────────────────────
nfam = len(results)
ncols = 3; nrows = (nfam + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 5*nrows))
axes = axes.flatten()

for i, (name, res) in enumerate(results.items()):
    ax = axes[i]
    ax.scatter(res["y_te"], res["preds"], alpha=0.15, s=5, c="#2E75B6")
    lim = [res["y_te"].min()-2, res["y_te"].max()+2]
    ax.plot(lim, lim, "r--", linewidth=1.2)
    ax.set_xlabel("Real [°C]", fontsize=9); ax.set_ylabel("Predicho [°C]", fontsize=9)
    ax.set_title(f"{name}  MAE={res['mae']:.2f}°C  R²={res['r2']:.4f}",
                 fontsize=9, fontweight="bold")
    ax.grid(alpha=0.3)

for j in range(i+1, len(axes)): axes[j].set_visible(False)

plt.suptitle("Predicho vs Real por familia de features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("familias_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── MAE por temperatura y familia (heatmap) ───────────────────────────────────
temps = sorted(df["temperature"].unique())
fams_no_all = [n for n in results if n != "Todas"]
mat = np.zeros((len(fams_no_all), len(temps)))
for i, fam in enumerate(fams_no_all):
    y_te = results[fam]["y_te"]; preds = results[fam]["preds"]
    for j, t in enumerate(temps):
        mask = y_te == t
        mat[i, j] = mean_absolute_error(y_te[mask], preds[mask]) if mask.sum() > 0 else np.nan

fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(mat, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(temps))); ax.set_xticklabels([str(int(t)) for t in temps], fontsize=8)
ax.set_yticks(range(len(fams_no_all))); ax.set_yticklabels(fams_no_all, fontsize=9)
for i in range(len(fams_no_all)):
    for j in range(len(temps)):
        if not np.isnan(mat[i,j]):
            ax.text(j, i, f"{mat[i,j]:.1f}", ha="center", va="center",
                    fontsize=6, color="black" if mat[i,j] < mat[~np.isnan(mat)].max()*0.6 else "white")
plt.colorbar(im, ax=ax, label="MAE [°C]", shrink=0.8)
ax.set_xlabel("Temperatura real [°C]", fontsize=10)
ax.set_title("Heatmap MAE por familia × temperatura", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("familias_heatmap_mae.png", dpi=150, bbox_inches="tight")
plt.show()